# RAG Pipeline Study Material

This notebook demonstrates the core pieces of a Retrieval-Augmented Generation system before moving into the live ChromaDB showcase:

- chunking strategies
- embeddings
- indexing
- retrieval and ranking
- response assembly

The first section explains the retrieval strategies in detail, including dense, sparse, hybrid, and reranking approaches. After that, the notebook keeps the existing PDF-based ChromaDB walkthrough in place.

## Retrieval strategies in RAG

RAG is not just “embed and search.” The retrieval layer controls what evidence reaches the model, so it is one of the main determinants of answer quality.

### 1. Embedded, or dense, retrieval

Dense retrieval embeds the query and the document chunks into the same vector space. ChromaDB then performs similarity search over those vectors. This is the strategy used in the showcase notebook.

Dense retrieval is strong when the user asks conceptual or paraphrased questions. For example, a query such as “How does the company make money?” can match chunks that mention revenue, sales, product lines, or data center demand even if the exact words do not appear.

Advantages:
- captures semantic similarity
- works well for natural-language questions
- usually gives better recall on long-form text

Limitations:
- may miss exact terms such as product names, tickers, IDs, or code tokens
- depends heavily on embedding quality
- can return semantically related but not exact matches

### 2. Separate, or sparse, retrieval

Sparse retrieval uses a different mechanism from embeddings. A classic example is BM25 or TF-IDF keyword search. The query and documents are represented by term statistics rather than semantic vectors.

Sparse retrieval is strong when exact wording matters. If the user asks for a specific ticker, table label, page title, or product code, keyword-based retrieval often finds the right evidence faster and more reliably than dense retrieval.

Advantages:
- excellent for exact-term matching
- transparent and easy to debug
- strong on named entities, rare terms, and identifiers

Limitations:
- weak at paraphrase and semantic matching
- vocabulary mismatch can hurt recall
- not as good at broad conceptual questions

### 3. Hybrid retrieval

Hybrid retrieval combines dense and sparse methods. This is often the best default for production systems because it covers both semantic and exact-match use cases.

A common pattern is:
1. run dense retrieval
2. run sparse retrieval
3. merge the results
4. rerank the merged list

This is useful when documents contain both natural language and structured details.

### 4. Two-stage retrieval and reranking

Two-stage retrieval first retrieves a broad candidate set, then reranks it with a stronger but slower model. The first stage prioritizes speed and recall. The second stage prioritizes precision.

This works well when you want better answer grounding but do not want to score every document with a heavy model.

### 5. Chunk-aware retrieval

Retrieval quality also depends on how chunks are built. If a chunk is too small, the retrieval step may find a fragment that lacks enough context. If a chunk is too large, the retrieved evidence can include irrelevant material and waste the context window.

The notebook uses overlapping chunks so a relevant sentence near a boundary still appears in at least one chunk.

### Practical rule of thumb

- Use dense retrieval for semantic Q&A.
- Use sparse retrieval when exact terms matter.
- Use hybrid retrieval when you want the most robust general-purpose setup.
- Add reranking when precision becomes important.

The showcase below keeps the current ChromaDB workflow in place, so you can see the full path from PDF ingestion to retrieval and answering.

## Why RAG is needed

A language model is strong at generating fluent text, but it is not a guaranteed source of truth. It can answer confidently even when it does not know the answer. RAG reduces that problem by attaching external evidence to the generation step.

RAG helps an LLM in four important ways:

- it gives the model access to fresh or private knowledge that is not in the base model weights
- it grounds the answer in retrieved context instead of relying only on memory
- it makes answers easier to trace back to source documents
- it reduces the amount of context you need to stuff into the prompt manually

In practice, RAG is useful whenever the answer depends on documents, policies, reports, manuals, tickets, or other changing content.

## Hallucination

Hallucination is when an LLM produces an answer that sounds plausible but is not supported by facts. This can happen because the model is trained to predict likely text, not to verify truth.

RAG reduces hallucination by narrowing the model’s attention to retrieved evidence. That does not remove hallucination completely, but it lowers the chance that the model will invent details when the right context is available.

A good RAG system still needs guardrails:

- retrieve the right chunks
- keep chunks small enough to stay relevant
- include source metadata
- tell the model to answer only from the context when possible

## Why indexing matters

Indexing is the step that makes retrieval fast enough to be practical. Without an index, the system would need to compare the query against every chunk every time, which becomes expensive as the corpus grows.

A vector index stores embeddings in a structure optimized for similarity search. That gives you:

- faster query time
- lower latency for interactive apps
- the ability to scale to many chunks or documents
- a stable persistence layer so you do not rebuild everything on every question

In this notebook, ChromaDB acts as the local persistent index in `vector_db/`.

## How embeddings work

Embeddings turn text into vectors, which are lists of numbers that capture meaning. Text that is semantically similar tends to map to nearby vectors. That is why a question like “How does the company make money?” can still retrieve chunks that mention revenue, demand, or product lines even if the exact wording differs.

The pipeline is usually:

1. normalize the text
2. send it to an embedding model
3. receive a vector
4. store the vector with the chunk
5. embed the query the same way
6. compare vectors with cosine similarity or a similar distance function

The important rule is that the query and the documents must use the same embedding space. If they do not, similarity search becomes meaningless.

### Python example: embedding text

The snippet below shows the shape of an embedding workflow using Ollama embeddings. The same pattern works with a GitHub-hosted embedding client if you swap the provider.

```python
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434/v1",
)

query_vector = embeddings.embed_query("What does the report say about revenue?")
document_vector = embeddings.embed_documents([
    "The report explains revenue, product demand, and business growth."
])

print(len(query_vector))
print(len(document_vector[0]))
print(query_vector[:10])
```

What matters here is not the exact numbers, but the fact that both query and document are projected into the same vector space.

## Batch chunking and why it matters

Batch chunking means splitting the source material into multiple smaller chunks and processing them in batches rather than loading one huge block of text into the prompt.

This is important for three reasons:

- it preserves local meaning better than one large dump of text
- it improves retrieval precision because each chunk is narrower and more focused
- it helps control token usage because only the most relevant chunks are sent to the model

In RAG systems, batch chunking is a practical optimization step. You do not want to send the whole document to the LLM every time. You want to retrieve a small set of high-value chunks, and ideally only the top few that are relevant to the question.

## Saving tokens in RAG

Token cost is one of the main reasons to use RAG carefully. The biggest savings come from reducing how much text reaches the LLM.

Common token-saving strategies include:

- smaller chunks with overlap only where needed
- top-k retrieval, where you return only the best few chunks
- metadata filters, such as restricting retrieval to a document, page, or section
- reranking, so you can retrieve broadly but send only the strongest matches to the model
- answer compression, where a smaller model or summarizer shrinks long retrieved text before the final prompt

The tradeoff is always the same: fewer tokens usually means lower cost and faster responses, but too aggressive a reduction can hurt answer quality.

A good workflow is: retrieve broadly, rerank, then send only the smallest context that still supports a grounded answer.

In [1]:
retrieval_strategies = {
    'embedded / dense retrieval': 'Convert both query and chunks to vectors and use similarity search in ChromaDB. Best when semantic matching matters more than exact keyword overlap.',
    'separate / sparse retrieval': 'Use a separate lexical retriever such as BM25 or keyword search. Best when exact terms, IDs, or code-like tokens matter.',
    'hybrid retrieval': 'Combine dense and sparse retrieval, then merge or rerank the results. Best general-purpose choice for mixed documents.',
    'two-stage retrieval': 'First retrieve broadly with a fast retriever, then rerank with a stronger model or cross-encoder. Good when precision matters.',
}

for name, description in retrieval_strategies.items():
    print(f'- {name}: {description}')

- embedded / dense retrieval: Convert both query and chunks to vectors and use similarity search in ChromaDB. Best when semantic matching matters more than exact keyword overlap.
- separate / sparse retrieval: Use a separate lexical retriever such as BM25 or keyword search. Best when exact terms, IDs, or code-like tokens matter.
- hybrid retrieval: Combine dense and sparse retrieval, then merge or rerank the results. Best general-purpose choice for mixed documents.
- two-stage retrieval: First retrieve broadly with a fast retriever, then rerank with a stronger model or cross-encoder. Good when precision matters.


# RAG with ChromaDB and a Real PDF

This notebook builds a practical Retrieval-Augmented Generation pipeline that:

- loads a PDF from `doc/`
- splits it into chunks with overlap
- embeds the chunks with local Ollama embeddings or a GitHub-hosted embedding model
- stores the vectors in a persistent ChromaDB database under `vector_db/`
- retrieves the most relevant chunks for a question
- sends the retrieved context to a chat model to answer the question

The showcase document in this lesson is `doc/NVDA_report.pdf`.

In [2]:
from __future__ import annotations

import os
import shutil
from pathlib import Path

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


def find_rag_directory() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        direct = candidate / 'doc' / 'NVDA_report.pdf'
        nested = candidate / 'ai_tutorials' / 'rag' / 'doc' / 'NVDA_report.pdf'
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / 'ai_tutorials' / 'rag'
    raise FileNotFoundError('Could not find ai_tutorials/rag/doc/NVDA_report.pdf')


RAG_DIR = find_rag_directory()
DOC_PATH = RAG_DIR / 'doc' / 'NVDA_report.pdf'
VECTOR_DB_DIR = RAG_DIR / 'vector_db'
COLLECTION_NAME = 'rag_nvda_report'

OLLAMA_ENDPOINT = 'http://192.168.1.188'
os.environ['OLLAMA_BASE_URL'] = f'{OLLAMA_ENDPOINT}:11434'

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
OLLAMA_ENDPOINT = os.getenv('OLLAMA_ENDPOINT', OLLAMA_ENDPOINT)
EMBEDDING_PROVIDER = os.getenv('RAG_EMBEDDING_PROVIDER', 'ollama').lower()
OLLAMA_EMBEDDING_MODEL = os.getenv('OLLAMA_EMBEDDING_MODEL', 'nomic-embed-text')

GITHUB_BASE_URL = os.getenv('GITHUB_MODELS_BASE_URL', 'https://models.inference.ai.azure.com')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
GITHUB_EMBEDDING_MODEL = os.getenv('GITHUB_EMBEDDING_MODEL', 'text-embedding-3-small')
GITHUB_CHAT_MODEL = os.getenv('GITHUB_MODEL', 'gpt-4o-mini')
RESET_VECTOR_DB = False

print('RAG directory:', RAG_DIR)
print('Document path:', DOC_PATH)
print('Vector DB path:', VECTOR_DB_DIR)
print('Embedding provider:', EMBEDDING_PROVIDER)


Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


RAG directory: /Users/yevgeniy/Development/Projects/GenerativeAI/ai-engineering/tutorials/rag
Document path: /Users/yevgeniy/Development/Projects/GenerativeAI/ai-engineering/tutorials/rag/doc/NVDA_report.pdf
Vector DB path: /Users/yevgeniy/Development/Projects/GenerativeAI/ai-engineering/tutorials/rag/vector_db
Embedding provider: ollama


In [ ]:

loader = PyPDFLoader(str(DOC_PATH))
page_documents = loader.load()
print(f'Loaded {len(page_documents)} pages from the PDF')
print(page_documents[0].page_content[:1000])

splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunk_documents = splitter.split_documents(page_documents)
for chunk_index, chunk in enumerate(chunk_documents):
    chunk.metadata['chunk_index'] = chunk_index
print(f'Created {len(chunk_documents)} chunks')
print(chunk_documents[0].metadata)
print(chunk_documents[0].page_content[:800])

Loaded 1 pages from the PDF
Equity Research Report: NVIDIA Corporation
Income Summarization
In fiscal year 2024, NVIDIA experienced a remarkable 126% year-over-year revenue
growth, reaching $60.9 billion, primarily driven by a 217% surge in its Compute &
Networking segment, which amounted to $47.405 billion, and a 15% increase in
Gaming revenue within the Graphics segment, totaling $13.517 billion. This growth
was underpinned by strong demand for data center systems and products, alongside
strategic product launches like the GeForce RTX 4060 and 4070 GPUs. The
company's gross margin improved significantly to 72.7%, with operating income
escalating by 681%, indicating efficient cost management and operational
excellence. Despite facing supply chain complexities and geopolitical risks,
particularly in China, NVIDIA's strategic focus on AI and accelerated computing
platforms, including NVIDIA DGX Cloud services and AI Foundations, has positioned
it for sustained growth and market leadersh

## Build the persistent ChromaDB index

This is the indexing step. The chunks are embedded and written into ChromaDB inside the local `vector_db/` folder.

If you want a clean rebuild, set `RESET_VECTOR_DB = True` before running the next cell.

### Dual-store design for optimization

This implementation uses two Chroma collections for the same document:

- detail collection: chunk-level embeddings for full retrieval coverage
- summary collection: one compact metadata summary for fast context priming

The summary store is cheap to retrieve and helps the model get document-level framing before reading detailed chunks.

In [ ]:
# OLLAMA_CHAT_MODEL = os.getenv('OLLAMA_MODEL', 'llama3.2')
OLLAMA_CHAT_MODEL = 'llama3.2'

In [ ]:
print('Ollama base url:', OLLAMA_BASE_URL)
print('Ollama endpoint:', OLLAMA_ENDPOINT)
print('Ollama embedding model:', OLLAMA_EMBEDDING_MODEL)
print('Ollama chat model:', OLLAMA_CHAT_MODEL)

Ollama base url: http://192.168.1.188:11434
Ollama endpoint: http://localhost:11434/v1
Ollama embedding model: nomic-embed-text
Ollama chat model: llama3.1


In [4]:
def build_embeddings():
    if EMBEDDING_PROVIDER == 'github':
        if not GITHUB_TOKEN:
            raise ValueError('GITHUB_TOKEN is required when RAG_EMBEDDING_PROVIDER=github')
        return OpenAIEmbeddings(
            model=GITHUB_EMBEDDING_MODEL,
            api_key=GITHUB_TOKEN,
            base_url=GITHUB_BASE_URL,
        )

    return OllamaEmbeddings(
        model=OLLAMA_EMBEDDING_MODEL,
        base_url=OLLAMA_BASE_URL,
    )


def build_document_summary_text(pages, chunks, doc_path: Path) -> str:
    preview = ' '.join(page.page_content.strip().replace('\n', ' ') for page in pages[:1])[:1000]
    return (
        f"Document: {doc_path.name}\n"
        f"Source path: {doc_path}\n"
        f"Total pages: {len(pages)}\n"
        f"Total chunks: {len(chunks)}\n"
        "Summary preview: "
        f"{preview}"
    )


embeddings = build_embeddings()
print(type(embeddings).__name__)

if RESET_VECTOR_DB and VECTOR_DB_DIR.exists():
    shutil.rmtree(VECTOR_DB_DIR)

VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

DETAIL_COLLECTION_NAME = f'{COLLECTION_NAME}_detail'
SUMMARY_COLLECTION_NAME = f'{COLLECTION_NAME}_summary'

# 1) Detailed chunk store for high-recall retrieval.
detail_store = Chroma.from_documents(
    documents=chunk_documents,
    embedding=embeddings,
    collection_name=DETAIL_COLLECTION_NAME,
    persist_directory=str(VECTOR_DB_DIR),
)

# 2) Metadata summary store for cheap document-level context.
summary_text = build_document_summary_text(page_documents, chunk_documents, DOC_PATH)
summary_store = Chroma(
    collection_name=SUMMARY_COLLECTION_NAME,
    persist_directory=str(VECTOR_DB_DIR),
    embedding_function=embeddings,
)
summary_store.delete(where={"doc_id": DOC_PATH.name})
summary_store.add_texts(
    texts=[summary_text],
    metadatas=[
        {
            "doc_id": DOC_PATH.name,
            "source": str(DOC_PATH),
            "pages": len(page_documents),
            "chunks": len(chunk_documents),
            "record_type": "document_summary",
        }
    ],
    ids=[f'{DOC_PATH.name}_summary'],
)

reloaded_detail_store = Chroma(
    collection_name=DETAIL_COLLECTION_NAME,
    persist_directory=str(VECTOR_DB_DIR),
    embedding_function=embeddings,
)
reloaded_summary_store = Chroma(
    collection_name=SUMMARY_COLLECTION_NAME,
    persist_directory=str(VECTOR_DB_DIR),
    embedding_function=embeddings,
)

print('Detail collection:', DETAIL_COLLECTION_NAME)
print('Detail chunk count:', reloaded_detail_store._collection.count())
print('Summary collection:', SUMMARY_COLLECTION_NAME)
print('Summary record count:', reloaded_summary_store._collection.count())
print('Vector DB stored in:', VECTOR_DB_DIR)

OllamaEmbeddings
Detail collection: rag_nvda_report_detail
Detail chunk count: 6
Summary collection: rag_nvda_report_summary
Summary record count: 1
Vector DB stored in: /Users/yevgeniy/Development/Projects/GenerativeAI/ai-engineering/tutorials/rag/vector_db


## Retrieve relevant context and ask a question

The query is embedded with the same model used for the document chunks. ChromaDB returns the closest chunks, which are then used as context for the final answer.

### Optimized retrieval workflow

This section implements a token-efficient strategy:

1. retrieve broadly from the detail store (`broad_k`)
2. rerank with a fused score (dense relevance + lexical overlap)
3. keep only the minimal evidence set under a token budget
4. prepend document-level summary metadata from the separate summary store

This follows the recommended workflow: retrieve broadly, rerank, then send the smallest grounded context.

In [5]:
import re


def retrieve_context(question: str, top_k: int = 4):
    results = reloaded_detail_store.similarity_search_with_score(question, k=top_k)
    lines = []
    for rank, (doc, score) in enumerate(results, start=1):
        source = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', 'unknown')
        chunk_index = doc.metadata.get('chunk_index', 'unknown')
        lines.append(
            f'{rank}. score={score:.4f} | source={source} | page={page} | chunk={chunk_index}\n{doc.page_content}'
        )
    return results, '\n\n'.join(lines)


question = 'What information does the report give about NVIDIA and its business?'
matches, context_text = retrieve_context(question, top_k=3)
print(context_text[:2500])


def build_chat_model():
    if EMBEDDING_PROVIDER == 'github':
        if not GITHUB_TOKEN:
            raise ValueError('GITHUB_TOKEN is required when RAG_EMBEDDING_PROVIDER=github')
        return ChatOpenAI(
            model=GITHUB_CHAT_MODEL,
            api_key=GITHUB_TOKEN,
            base_url=GITHUB_BASE_URL,
            temperature=0,
        )

    return ChatOllama(
        model=OLLAMA_CHAT_MODEL,
        base_url=OLLAMA_BASE_URL,
        temperature=0,
    )


chat_model = build_chat_model()
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You answer questions using only the provided context. If the context is insufficient, say so clearly.'),
    ('human', 'Question:\n{question}\n\nContext:\n{context}'),
])


def answer_question(question: str, top_k: int = 4):
    _, context = retrieve_context(question, top_k=top_k)
    chain = prompt | chat_model
    return chain.invoke({'question': question, 'context': context})


response = answer_question('Summarize the key point of the document in one paragraph.', top_k=3)
print(response.content if hasattr(response, 'content') else response)


def normalize_tokens(text: str) -> set[str]:
    return set(re.findall(r'[a-z0-9]+', text.lower()))


def rerank_results(question: str, dense_results):
    question_tokens = normalize_tokens(question)
    ranked = []
    for dense_rank, (doc, distance) in enumerate(dense_results, start=1):
        # Chroma distance is lower-is-better; convert to a bounded relevance score.
        dense_relevance = 1.0 / (1.0 + float(distance))
        doc_tokens = normalize_tokens(doc.page_content)
        overlap = len(question_tokens & doc_tokens)
        # Weighted fusion between dense similarity and lexical overlap.
        fused_score = (0.75 * dense_relevance) + (0.25 * overlap)
        ranked.append((fused_score, dense_rank, doc, distance, overlap))

    ranked.sort(key=lambda item: item[0], reverse=True)
    return ranked


def build_minimal_context(reranked, token_budget: int = 900, max_chunks: int = 4) -> tuple[list, str]:
    selected = []
    token_count = 0

    for fused_score, dense_rank, doc, distance, overlap in reranked:
        chunk_tokens = len(doc.page_content.split())
        if selected and token_count + chunk_tokens > token_budget:
            continue

        selected.append((fused_score, dense_rank, doc, distance, overlap, chunk_tokens))
        token_count += chunk_tokens

        if len(selected) >= max_chunks or token_count >= token_budget:
            break

    context_parts = []
    for idx, (fused_score, dense_rank, doc, distance, overlap, chunk_tokens) in enumerate(selected, start=1):
        source = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', 'unknown')
        chunk_index = doc.metadata.get('chunk_index', 'unknown')
        context_parts.append(
            f'[{idx}] fused={fused_score:.4f} dense_distance={distance:.4f} overlap={overlap} '
            f'source={source} page={page} chunk={chunk_index}\n{doc.page_content}'
        )

    return selected, '\n\n'.join(context_parts)


def get_document_summary_for_question(question: str) -> str:
    summary_hits = reloaded_summary_store.similarity_search(question, k=1)
    return summary_hits[0].page_content if summary_hits else 'No document summary available.'


def answer_question_optimized(
    question: str,
    broad_k: int = 12,
    token_budget: int = 900,
    max_chunks: int = 4,
):
    # Step 1: retrieve broadly for recall.
    dense_results = reloaded_detail_store.similarity_search_with_score(question, k=broad_k)

    # Step 2: rerank to improve precision.
    reranked = rerank_results(question, dense_results)

    # Step 3: keep the smallest grounded context under a token budget.
    selected, minimal_context = build_minimal_context(
        reranked,
        token_budget=token_budget,
        max_chunks=max_chunks,
    )

    # Step 4: prepend document-level summary metadata from separate summary store.
    doc_summary = get_document_summary_for_question(question)
    final_context = (
        'Document summary metadata:\n'
        f'{doc_summary}\n\n'
        'Selected detailed evidence:\n'
        f'{minimal_context}'
    )

    chain = prompt | chat_model
    answer = chain.invoke({'question': question, 'context': final_context})

    diagnostics = {
        'broad_retrieval_count': len(dense_results),
        'selected_chunk_count': len(selected),
        'selected_token_estimate': sum(item[5] for item in selected),
        'token_budget': token_budget,
    }

    return answer, diagnostics, final_context


optimized_response, optimized_diagnostics, optimized_context = answer_question_optimized(
    question='What are the key business highlights in this report?',
    broad_k=12,
    token_budget=700,
    max_chunks=3,
)

print('Diagnostics:', optimized_diagnostics)
print('Optimized answer:\n')
print(optimized_response.content if hasattr(optimized_response, 'content') else optimized_response)
print('\nContext preview:\n')
print(optimized_context[:2200])

1. score=0.4692 | source=/Users/yevgeniy/Development/Projects/GenerativeAI/ai-engineering/tutorials/rag/doc/NVDA_report.pdf | page=0 | chunk=0
Equity Research Report: NVIDIA Corporation
Income Summarization
In fiscal year 2024, NVIDIA experienced a remarkable 126% year-over-year revenue
growth, reaching $60.9 billion, primarily driven by a 217% surge in its Compute &
Networking segment, which amounted to $47.405 billion, and a 15% increase in
Gaming revenue within the Graphics segment, totaling $13.517 billion. This growth
was underpinned by strong demand for data center systems and products, alongside
strategic product launches like the GeForce RTX 4060 and 4070 GPUs. The
company's gross margin improved significantly to 72.7%, with operating income
escalating by 681%, indicating efficient cost management and operational
excellence. Despite facing supply chain complexities and geopolitical risks,
particularly in China, NVIDIA's strategic focus on AI and accelerated computing

2. score=

ResponseError: model 'llama3.1' not found (status code: 404)

## What was built

This notebook now contains a working RAG pipeline. It loads a real PDF, splits it into chunks, embeds those chunks, stores them in persistent ChromaDB, and uses retrieval plus prompting to answer a user question.

The main idea is that the model is no longer asked to remember everything. The retriever finds evidence in `vector_db/`, and the chat model uses that evidence to produce the answer.